In [1]:
import pandas as pd

In [2]:
# df = pd.read_csv("data/customer_churn_dataset-training-master.csv")
df = pd.read_csv("data/Bank_Churn.csv")


In [4]:
df

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [5]:
features = ['CreditScore', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
target = 'Exited'

In [12]:
ir_after_rebalance = 1.0  # Placeholder value, replace with actual imbalance ratio after rebalancing

ir_tr = float((df[target].values == 0).sum() / (df[target].values == 1).sum())
model_rebalance = ir_tr / ir_after_rebalance

n0, n1 = (df[target].values == 0).sum(), (df[target].values == 1).sum()
mayoritary_label = 0.0 if n0 >= n1 else 1.0

## NN

In [ ]:
import torch
from utils import MonotonicNN

In [ ]:
# ### NN params ### #
# "learning_rate": 0.0001,
# "momentum": 0.0,
# "centered": False,
# "num_epochs": 100,
# "batch_size": 1000,
# "hidden_size_non_monotonic": 16,
# "hidden_size_positive_monotonic": 8,
# "hidden_size_negative_monotonic": 8

In [ ]:
all_vars = ["monthly_price", "price_increase_3m", "discount_pct", "tenure_months", "tickets_last90d"]
pos_vars = ["monthly_price", "price_increase_3m"]   # ↑ price ⇒ ↑ churn prob
neg_vars = ["discount_pct"]                         # ↑ discount ⇒ ↓ churn prob
non_vars = [c for c in all_vars if c not in pos_vars + neg_vars]

model = MonotonicNN(
    all_variables=all_vars,
    non_monotonic_vars=non_vars,
    positive_monotonic_vars=pos_vars,
    negative_monotonic_vars=neg_vars,
    hidden_non=16, hidden_pos=8, hidden_neg=8
)

In [ ]:
# ### Cast data into torch tensor ### #
x_tr = torch.from_numpy(df[features].values)
y_tr = torch.from_numpy(df[target].values)

In [ ]:
# RMSprop, maybe Adam is an option
optimizer = torch.optim.RMSprop(model.parameters(),
                                lr=1e-3,
                                momentum=0,
                                centered=False)

dataset_train = torch.utils.data.TensorDataset(x_tr, y_tr)
trainloader_train = torch.utils.data.DataLoader(dataset_train,
                                                batch_size=1024,
                                                shuffle=True,
                                                num_workers=0)

In [ ]:
for epoch in range(5):
    for xb, yb in trainloader_train:
        xb = xb.float()
        yb = yb.float().view(-1, 1)

        # rebalance_weights = self.compute_weights(targets=target, mayoritary_label=mayoritary_label)
        optimizer.zero_grad()
        p = model.forward(xb)                                # [B,1], probabilities
        loss = torch.mean((p - yb)**2)               # simple MSE; replace with weighted MSE if needed
        # weighted_mse_loss = torch.div(torch.sum(weights*((p - yb) ** 2)), torch.sum(weights))
        loss.backward()
        optimizer.step()
        model.monotonic_constraint()                 # enforce monotonicity after each step